# Nested Learning for CAD — W7 CPU report

Accelerator: None. Inputs: W7 source and all saved W7 evidence ZIPs. No dataset/ViT/W6 input needed.


In [ ]:

import os, sys, time, json, hashlib, zipfile, tarfile, shutil, subprocess
from pathlib import Path
if 'STARTED' not in globals():STARTED=time.monotonic()
sys.dont_write_bytecode=True
os.environ['PYTHONDONTWRITEBYTECODE']='1'
INPUT=Path('/kaggle/input')
SOURCE_OVERRIDE=None  # Source ZIP or Kaggle-extracted source directory.
W6_OVERRIDE=None      # Reviewed completed W6 v4 ZIP (NOT a preflight/paused W6).
RESUME_OVERRIDE=None  # W7 preflight or latest W7 output for this job.
C1_EVIDENCE_OVERRIDE=None  # A1 seed1/2: C1 output of that same seed.
MVTEC_OVERRIDE=None
CHECKPOINT_OVERRIDE=None
SESSION_HOURS=10.5
REPO=Path('/tmp/hope_w7_source')
W6=Path('/tmp/hope_w7_parent_w6')
WORK=Path('/kaggle/working/hope_w7')
ASSETS=Path('/tmp/hope_w7_assets')
EXPECTED_BACKBONE='2c4839589adbbf701573dd155e1814f8adb145cd4d235bb86a93269369e111ec'
def sha(path):
    with Path(path).open('rb') as stream:return hashlib.file_digest(stream,'sha256').hexdigest()
def unique(values,label):
    values=sorted(set(Path(p) for p in values))
    if len(values)!=1:raise RuntimeError(f'{label}: expected ONE input, got {values}; set its override')
    return values[0]
def safe(root,name):
    path=(root/name).resolve()
    if Path(name).is_absolute() or chr(92) in name or ':' in name or not path.is_relative_to(root.resolve()):
        raise RuntimeError('Unsafe archive path')
    return path
sources=[p.parent for p in INPUT.rglob('bundle_manifest.json')
         if json.loads(p.read_text()).get('schema')=='hope_w7_source_v1']
sources=sources or list(INPUT.rglob('hope_cad_w7_source_bundle.zip'))
source=Path(SOURCE_OVERRIDE) if SOURCE_OVERRIDE else unique(sources,'W7 source')
if source.is_dir():
    manifest=json.loads((source/'bundle_manifest.json').read_text())
    for name,expected in manifest['files'].items():
        original=source/name
        if sha(original)!=expected:raise RuntimeError('Source checksum mismatch: '+name)
        path=safe(REPO,name);path.parent.mkdir(parents=True,exist_ok=True)
        if path.exists() and sha(path)!=expected:raise RuntimeError('Working source changed: '+name)
        shutil.copyfile(original,path)
else:
    with zipfile.ZipFile(source) as archive:
        manifest=json.loads(archive.read('bundle_manifest.json'))
        if len(archive.namelist())!=len(set(archive.namelist())):raise RuntimeError('Duplicate source entries')
        for name,expected in manifest['files'].items():
            raw=archive.read(name)
            if hashlib.sha256(raw).hexdigest()!=expected:raise RuntimeError('Source checksum mismatch: '+name)
            path=safe(REPO,name);path.parent.mkdir(parents=True,exist_ok=True)
            if path.exists() and sha(path)!=expected:raise RuntimeError('Working source changed: '+name)
            path.write_bytes(raw)
if manifest['schema']!='hope_w7_source_v1':raise RuntimeError('Wrong source bundle')
(REPO/'bundle_manifest.json').write_text(json.dumps(manifest))
os.chdir(REPO);sys.path.insert(0,str(REPO))
print('Verified source files:',len(manifest['files']),'origin:',manifest['origin'])

EVIDENCE=sorted(INPUT.rglob('hope_cad_w7_evidence.zip'))
REPORT=Path('/kaggle/working/hope_w7_report')
INVENTORY_ONLY=True  # First inspect missing jobs; set False for complete paired uncertainty.
EXACT_PIXELS=False   # Only histogram bootstrap is approximate, with strict exact validation.
from training.hope_w7_io import restore_archive
old=list(INPUT.rglob('hope_cad_w7_analysis.zip'))
if old:restore_archive(unique(old,'Previous CPU analysis'),REPORT,schema='hope_w7_analysis_v1')


In [ ]:

command=[sys.executable,'-B','scripts/hope_cad_v2/report_w7.py','--evidence',*map(str,EVIDENCE),
         '--output-dir',str(REPORT),'--session-hours',str(SESSION_HOURS)]
if INVENTORY_ONLY:command+=['--inventory-only']
if EXACT_PIXELS:command+=['--exact-pixels']
print(' '.join(command));EXIT_CODE=subprocess.call(command,cwd=REPO)
print('Report exit code:',EXIT_CODE)


In [ ]:

from training.hope_w7_io import export_archive
from IPython.display import display,FileLink
result=export_archive(REPORT,Path('/kaggle/working/hope_cad_w7_analysis.zip'),schema='hope_w7_analysis_v1')
print(result)
if (REPORT/'handoff.json').exists():print((REPORT/'handoff.json').read_text())
display(FileLink(result['path']))
